# 05 · Main method, step 1: open language models write the plans

Each question goes to an open model through the Hugging Face router (OpenAI-compatible API), with the query
language (`configs/language.md`) and the catalogue as the system prompt: temperature 0, one request per question
and wording, every request and reply saved in `outputs/llm/` (a saved reply is reused, never re-asked). A reply
that is not one JSON object, or a plan that fails validation, is a **malformed plan**: counted, scored F1 0.

Wordings: **as written** and **reworded by hand** in the seen areas; **as written** with the state name changed in
the held-out states. Models (pinned with their provider): main `openai/gpt-oss-120b:groq`; information:
`zai-org/GLM-5.3:nscale` (large, newer) and `openai/gpt-oss-20b:groq` (small, can run on the Mac).

Each plan is scored twice against the truth (notebook 04): on the **official maps** (translation errors only) and on
the **AlphaEarth maps** (end to end).

In [1]:
import json
import time

import numpy as np
import pandas as pd

from askmap.catalogue import area_classes
from askmap.config import PROJECT_ROOT, data_root, load_config
from askmap.layers import SOLAR_LATER, official_leaf_map
from askmap.plan import AreaContext, execute, validate
from askmap.power import add_derived, fetch_daily, pixel_cells
from askmap.score import f1
from askmap.tiles import area_tiles, pixel_lonlat
from askmap.universe import Universe, box_of

cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
Q = load_config(PROJECT_ROOT / "configs" / "queries.yaml")["queries"]
RW = load_config(PROJECT_ROOT / "configs" / "reworded.yaml")["reworded"]
ROOT = data_root()
RES = PROJECT_ROOT / "outputs" / "results"
FIG = PROJECT_ROOT / "outputs" / "figures"
CL, P, SC = cfg["classes"], cfg["power"], cfg["scoring"]
LEAVES, GROUPS = CL["leaves"], CL["groups"]
CATALOGUE = area_classes(cfg)
T_START = time.time()
cells = pd.read_parquet(ROOT / "power" / "universe_cells.parquet")
daily = {
    box: add_derived(fetch_daily(P, b, ROOT / "power" / f"universe_{box}_daily.nc"), P["low_wind_sun_pct"])
    for box, b in P["boxes"].items()
}
uni = Universe(cells=cells, daily=daily)
SEEN = ["north_dakota", "north_carolina"]
HELD = cfg["heldout_pools"]  # seen area -> held-out state
POOL_Q = {
    "north_dakota": "north_dakota",
    "north_carolina": "north_carolina",
    "minnesota": "north_dakota",
    "georgia": "north_carolina",
}  # pool -> area whose questions it answers
T = {}
for area in POOL_Q:
    for name, (tile, role) in area_tiles(cfg["areas"][area], cfg["grid"]).items():
        z = np.load(ROOT / "tiles" / f"{area}_{name}.npz")
        solar = z["solar"] if "solar" in z.files else None
        nwi = z["nwi"] if "nwi" in z.files else None
        valid = np.isfinite(z["ae"]).all(axis=0)
        if solar is not None:
            valid &= solar != SOLAR_LATER
        lon, lat = pixel_lonlat(tile)
        box = box_of(float(lon.mean()), float(lat.mean()), P["boxes"])
        iy, ix = pixel_cells(daily[box].lat.values, daily[box].lon.values, lat, lon)
        T[name] = {
            "area": area,
            "role": role,
            "valid": valid,
            "box": box,
            "iy": iy,
            "ix": ix,
            "cell": uni.index(box, iy, ix),
            "official": official_leaf_map(z["cdl"], LEAVES, CL, nwi=nwi, solar=solar),
        }
        if role == "test":
            T[name]["ae_map"] = np.load(ROOT / "maps" / f"ae_{name}.npz")["leaf"]
            T[name]["planted_map"] = np.load(ROOT / "maps" / f"planted_{name}.npz")["leaf"]
CTX, TILES = {}, {}
for area in POOL_Q:
    names = [n for n, t in T.items() if t["area"] == area]
    ds = daily[T[names[0]]["box"]]
    ac = np.zeros((ds.sizes["lat"], ds.sizes["lon"]), bool)
    for n in names:
        ac[T[n]["iy"], T[n]["ix"]] = True
    CTX[area] = AreaContext(
        leaves=LEAVES,
        groups=GROUPS,
        res_m=cfg["grid"]["res_m"],
        power=ds,
        area_cells=ac,
        pixel_cell={n: (T[n]["iy"], T[n]["ix"]) for n in names},
    )
    TILES[area] = {
        "train": [n for n in names if T[n]["role"] == "train"],
        "test": [n for n in names if T[n]["role"] == "test"],
    }


def answer(plan, pool, maps="official"):
    """Answer of a plan in a pool's test tiles, as one vector over their valid pixels (fixed tile order)."""
    tiles = TILES[pool]["test"]
    key = "official" if maps == "official" else f"{maps}_map"
    got = execute(plan, {n: T[n][key] for n in tiles}, CTX[pool])
    return np.concatenate([got[n][T[n]["valid"]] for n in tiles])


def question_text(q, wording, pool):
    text = q["question"] if wording != "reworded" else RW[q["id"]]
    if pool in ("minnesota", "georgia"):
        text = text.replace("North Dakota", "Minnesota").replace("North Carolina", "Georgia")
    return text


def n_extra(plan):
    return len(plan["near"]) + len(plan["climate"]) + int(plan["land"]["min_patch_ha"] is not None)


print(f"{len(T)} tiles loaded; pools {dict((a, TILES[a]['test']) for a in POOL_Q)}")

18 tiles loaded; pools {'north_dakota': ['Kidder', 'Foster', 'Eddy'], 'north_carolina': ['NC4', 'NC5', 'NC6'], 'minnesota': ['MN1', 'MN2', 'MN3'], 'georgia': ['GA1', 'GA2', 'GA3']}


In [2]:
from concurrent.futures import ThreadPoolExecutor

from askmap.catalogue import catalogue_text
from askmap.llm import make_client, same_plan, system_prompt, translate

LLM = cfg["llm"]
OUT = PROJECT_ROOT / "outputs" / "llm"
OUT.mkdir(parents=True, exist_ok=True)
SYSTEM = system_prompt((PROJECT_ROOT / "configs" / "language.md").read_text(), catalogue_text(cfg))
(OUT / "system_prompt.md").write_text(SYSTEM)
client = make_client(LLM["base_url"])  # token from the local Hugging Face login; never printed
TRUTH = dict(np.load(ROOT / "maps" / "truth.npz"))
TASKS = []
for q in Q:
    seen = q["plan"]["area"]
    TASKS += [(q, "as_written", seen), (q, "reworded", seen), (q, "as_written", HELD[seen])]
print(f"{len(TASKS)} requests per model; system prompt {len(SYSTEM):,} characters")

174 requests per model; system prompt 6,647 characters


In [3]:
RECORDS = {}
for arm, model in LLM["models"].items():
    t0 = time.time()

    def ask(task, arm=arm, model=model):
        q, wording, pool = task
        path = OUT / arm / f"{wording}_{pool}_{q['id']}.json"
        return task, translate(client, model, SYSTEM, question_text(q, wording, pool), path, LLM)

    with ThreadPoolExecutor(LLM["workers"]) as pool_ex:
        RECORDS[arm] = list(pool_ex.map(ask, TASKS))
    secs = [r["seconds"] for _, r in RECORDS[arm]]
    provs = pd.Series([r["provider"] for _, r in RECORDS[arm]]).value_counts().to_dict()
    models = pd.Series([r["model_returned"] for _, r in RECORDS[arm]]).value_counts().to_dict()
    print(
        f"{arm}: {time.time() - t0:.0f}s wall; median {np.median(secs):.1f}s per request; providers {provs}; "
        f"models returned {models}"
    )

main: 0s wall; median 1.5s per request; providers {'groq': 174}; models returned {'openai/gpt-oss-120b': 174}
large: 0s wall; median 2.0s per request; providers {'nscale': 174}; models returned {'zai-org/GLM-5.3': 174}
small: 0s wall; median 1.4s per request; providers {'groq': 174}; models returned {'openai/gpt-oss-20b': 174}


## Score every plan: on the official maps (translation) and on the AlphaEarth maps (end to end)

In [4]:
rows = []
for arm, recs in RECORDS.items():
    for (q, wording, pool), rec in recs:
        ref = dict(q["plan"], area=pool)
        g = np.unpackbits(TRUTH[f"{q['id']}|{pool}"])[
            : sum(int(T[n]["valid"].sum()) for n in TILES[pool]["test"])
        ]
        g = g.astype(bool)
        row = {
            "arm": arm,
            "id": q["id"],
            "wording": wording,
            "pool": pool,
            "heldout": pool not in SEEN,
            "persona": q["persona"],
            "n_extra": n_extra(q["plan"]),
            "has_climate": bool(q["plan"]["climate"]),
            "truth_px": int(g.sum()),
            "error": rec["error"],
            "finish": rec["finish_reason"],
            "out_tokens": (rec["response"].get("usage") or {}).get("completion_tokens"),
        }
        plan = rec["plan"]
        if plan is not None:
            try:
                validate(plan, CATALOGUE, (2001, 2025))
            except (ValueError, KeyError, TypeError) as e:
                row["error"], plan = f"invalid plan: {e}", None
        if plan is None:
            row.update(F1_translation=0.0, F1_end_to_end=0.0, exact=False, wrong_area=False)
        elif plan["area"] != pool:
            row.update(F1_translation=0.0, F1_end_to_end=0.0, exact=False, wrong_area=True)
        else:
            row.update(
                F1_translation=f1(answer(plan, pool), g),
                F1_end_to_end=f1(answer(plan, pool, "ae"), g),
                exact=same_plan(plan, ref),
                wrong_area=False,
            )
        rows.append(row)
tr = pd.DataFrame(rows)
tr["scored"] = tr.truth_px >= SC["min_truth"]
tr.to_csv(RES / "05_translation.csv", index=False)
summary = tr.groupby(["arm", "wording", "heldout"]).agg(
    n=("id", "size"),
    malformed=("error", lambda s: int(s.notna().sum())),
    wrong_area=("wrong_area", "sum"),
    exact_plan=("exact", "mean"),
    median_F1_translation=("F1_translation", "median"),
    median_F1_end_to_end=("F1_end_to_end", "median"),
)
summary.round(3)

n  malformed  wrong_area  exact_plan  \
arm   wording    heldout                                          
large as_written False    58          2           0       0.897   
                 True     58          1           0       0.914   
      reworded   False    58          2           0       0.879   
main  as_written False    58          2           0       0.845   
                 True     58          1           0       0.862   
      reworded   False    58          1           0       0.862   
small as_written False    58          2           0       0.862   
                 True     58          2           0       0.879   
      reworded   False    58          1           0       0.845   

                          median_F1_translation  median_F1_end_to_end  
arm   wording    heldout                                               
large as_written False                      1.0                 0.763  
                 True                       1.0                 0.618  
      reworded   False                      1.0                 0.748  
main  as_written False                      1.0                 0.763  
                 True                       1.0                 0.618  
      reworded   False                      1.0                 0.751  
small as_written False                      1.0                 0.748  
                 True                       1.0                 0.618  
      reworded   False                      1.0                 0.676

## Malformed or invalid plans, and the main model's worst translations

In [5]:
pd.set_option("display.max_colwidth", 200)
print(tr[tr.error.notna()][["arm", "wording", "pool", "id", "error"]].to_string())

       arm     wording            pool    id                                                                       error
60    main  as_written    north_dakota  nd21                               invalid plan: climate[0]: op must be >= or <=
61    main    reworded    north_dakota  nd21                               invalid plan: climate[0]: op must be >= or <=
62    main  as_written       minnesota  nd21                               invalid plan: climate[0]: op must be >= or <=
126   main  as_written  north_carolina  nc11                               invalid plan: climate[0]: op must be >= or <=
177  large  as_written    north_dakota  nd02  malformed plan: JSONDecodeError: Expecting value: line 1 column 1 (char 0)
259  large    reworded    north_dakota  nd29  malformed plan: JSONDecodeError: Expecting value: line 1 column 1 (char 0)
300  large  as_written  north_carolina  nc11                               invalid plan: climate[0]: op must be >= or <=
301  large    reworded  north_ca

In [6]:
qmap = {q["id"]: q for q in Q}
bad = tr[(tr.arm == "main") & tr.scored & (tr.F1_translation < 0.9)].sort_values("F1_translation")
for r in bad.itertuples():
    path = OUT / "main" / f"{r.wording}_{r.pool}_{r.id}.json"
    plan = json.loads(path.read_text())["plan"]
    text = question_text(qmap[r.id], r.wording, r.pool)
    print(f"--- {r.id} {r.wording} {r.pool}: F1 {r.F1_translation:.2f}  {text}")
    print("model:    ", json.dumps(plan))
    print("reference:", json.dumps(dict(qmap[r.id]["plan"], area=r.pool)))

--- nd21 as_written north_dakota: F1 0.00  In the North Dakota study area, show sunflower fields that had fewer than 1,300 growing degree days (base 10 C) from May to September 2019.
model:     {"area": "north_dakota", "land": {"classes": ["sunflower"], "min_patch_ha": null}, "near": [], "climate": [{"variable": "GDD10", "years": [2019, 2019], "months": [5, 6, 7, 8, 9], "stat": "total", "day_threshold": null, "test": {"kind": "absolute", "op": "<", "value": 1300}}], "output": "map"}
reference: {"area": "north_dakota", "land": {"classes": ["sunflower"], "min_patch_ha": null}, "near": [], "climate": [{"variable": "GDD10", "years": [2019, 2019], "months": [5, 6, 7, 8, 9], "stat": "total", "day_threshold": null, "test": {"kind": "absolute", "op": "<=", "value": 1300}}], "output": "map"}
--- nd21 reworded north_dakota: F1 0.00  Sunflowers in the North Dakota area that accumulated under 1300 growing degree days (base 10 C) between May and September 2019.
model:     {"area": "north_dakota", "

In [7]:
print(f"Run time {(time.time() - T_START) / 60:.1f} min")

Run time 0.5 min
